In [0]:

# Creates the Silver schema 

### 0. Storage credential / external location check
Run this cell FIRST, before anything else in this notebook. It tells
you whether Bronze is backed by an explicit ADLS external location
(in which case Silver/Gold need their own, same pattern) or is plain
Unity Catalog-managed storage (in which case a plain CREATE SCHEMA is
all you need - no external location work at all).

In [0]:
display(spark.sql("SHOW STORAGE CREDENTIALS"))

name,comment
access-connector,null
dbw_pe_platform,null


In [0]:
display(spark.sql("SHOW EXTERNAL LOCATIONS"))

name,url,comment
bronze,abfss://bronze@pestorage3.dfs.core.windows.net/,null
dbw_pe_platform,abfss://unity-catalog-storage@dbstoragekprely4uekbbk.dfs.core.windows.net/7405615699402403,null
ingested,abfss://ingested@pestorage3.dfs.core.windows.net/,null


In [0]:
display(dbutils.fs.ls("abfss://bronze@pestorage3.dfs.core.windows.net/"))

path,name,size,modificationTime
abfss://bronze@pestorage3.dfs.core.windows.net/_quarantine/,_quarantine/,0,1789978262000
abfss://bronze@pestorage3.dfs.core.windows.net/commitment/,commitment/,0,1789978159000
abfss://bronze@pestorage3.dfs.core.windows.net/external_cash/,external_cash/,0,1789978299000
abfss://bronze@pestorage3.dfs.core.windows.net/external_payment/,external_payment/,0,1789978371000
abfss://bronze@pestorage3.dfs.core.windows.net/external_position/,external_position/,0,1789978257000
abfss://bronze@pestorage3.dfs.core.windows.net/external_reference/,external_reference/,0,1789978335000
abfss://bronze@pestorage3.dfs.core.windows.net/fund/,fund/,0,1789978130000
abfss://bronze@pestorage3.dfs.core.windows.net/investor/,investor/,0,1789978069000
abfss://bronze@pestorage3.dfs.core.windows.net/market_price/,market_price/,0,1789978034000
abfss://bronze@pestorage3.dfs.core.windows.net/payment/,payment/,0,1789975673000


In [0]:
CATALOG_CHECK = "dbw_pe_platform"  
display(spark.sql(f"DESCRIBE SCHEMA EXTENDED {CATALOG_CHECK}.bronze"))

database_description_item,database_description_value
Catalog Name,dbw_pe_platform
Namespace Name,bronze
Comment,
Location,abfss://bronze@pestorage3.dfs.core.windows.net/__unitystorage/schemas/17b5aa2a-c981-4799-8abf-fccfb50ede99
Owner,yash.singh_cs23@gla.ac.in
RootLocation,abfss://bronze@pestorage3.dfs.core.windows.net/
Properties,"((unity.catalog.managed.iceberg.defaults.delta.feature.catalogManaged,supported))"
Predictive Optimization,ENABLE (inherited from METASTORE metastore_azure_centralindia)


### 1. Confirm your catalog exists and you can see Bronze


In [0]:
CATALOG = "dbw_pe_platform"  

display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))
display(spark.sql(f"SHOW TABLES IN {CATALOG}.bronze"))

databaseName
bronze
default
information_schema
silver


database,tableName,isTemporary
bronze,commitment,false
bronze,external_cash,false
bronze,external_payment,false
bronze,external_position,false
bronze,external_reference,false
bronze,fund,false
bronze,investor,false
bronze,market_price,false
bronze,payment,false
bronze,portfolio_company,false


### 2.  Silver schema


In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver
COMMENT 'Silver layer - cleaned, deduplicated, DQ-checked, crosswalk-resolved data'
MANAGED LOCATION 'abfss://silver@pestorage3.dfs.core.windows.net/'
""")

print(f"Schema {CATALOG}.silver ready.")

Schema dbw_pe_platform.silver ready.


### 4. Pre-create the shared DQ log table
Every Silver notebook's `log_dq()` call appends here - creating the
table explicitly up front (rather than letting the first append
auto-create it) keeps the schema consistent across all 10 sources.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.dq_log (
    source_name      STRING,
    business_date     STRING,
    rule_name         STRING,
    records_checked   BIGINT,
    records_failed    BIGINT,
    reason_code       STRING,
    created_at        STRING
) USING DELTA
""")

print(f"{CATALOG}.silver.dq_log ready.")

dbw_pe_platform.silver.dq_log ready.


### 5. Sanity check
All 10 Bronze source tables should be visible

In [0]:
expected_sources = [
    "investor", "fund", "commitment", "portfolio_company", "payment", "market_price",
    "external_position", "external_cash", "external_reference", "external_payment",
]

for src in expected_sources:
    try:
        cnt = spark.table(f"{CATALOG}.bronze.{src}").count()
        print(f"OK   {src:<20} {cnt} rows")
    except Exception as e:
        print(f"MISSING  {src:<20} -> {e}")

OK   investor             250 rows
OK   fund                 25 rows
OK   commitment           330 rows
OK   portfolio_company    150 rows
OK   payment              222 rows
OK   market_price         1045 rows
OK   external_position    21 rows
OK   external_cash        10 rows
OK   external_reference   19 rows
OK   external_payment     15 rows


In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS dbw_pe_platform.bronze
MANAGED LOCATION 'abfss://bronze@pestorage3.dfs.core.windows.net/';

CREATE TABLE IF NOT EXISTS dbw_pe_platform.bronze.fund
USING DELTA LOCATION 'abfss://bronze@pestorage3.dfs.core.windows.net/fund/';

CREATE TABLE IF NOT EXISTS dbw_pe_platform.bronze.investor
USING DELTA LOCATION 'abfss://bronze@pestorage3.dfs.core.windows.net/investor/';

CREATE TABLE IF NOT EXISTS dbw_pe_platform.bronze.commitment
USING DELTA LOCATION 'abfss://bronze@pestorage3.dfs.core.windows.net/commitment/';

CREATE TABLE IF NOT EXISTS dbw_pe_platform.bronze.portfolio_company
USING DELTA LOCATION 'abfss://bronze@pestorage3.dfs.core.windows.net/portfolio_company/';

CREATE TABLE IF NOT EXISTS dbw_pe_platform.bronze.payment
USING DELTA LOCATION 'abfss://bronze@pestorage3.dfs.core.windows.net/payment/';

CREATE TABLE IF NOT EXISTS dbw_pe_platform.bronze.market_price
USING DELTA LOCATION 'abfss://bronze@pestorage3.dfs.core.windows.net/market_price/';

CREATE TABLE IF NOT EXISTS dbw_pe_platform.bronze.external_position
USING DELTA LOCATION 'abfss://bronze@pestorage3.dfs.core.windows.net/external_position/';

CREATE TABLE IF NOT EXISTS dbw_pe_platform.bronze.external_cash
USING DELTA LOCATION 'abfss://bronze@pestorage3.dfs.core.windows.net/external_cash/';

CREATE TABLE IF NOT EXISTS dbw_pe_platform.bronze.external_reference
USING DELTA LOCATION 'abfss://bronze@pestorage3.dfs.core.windows.net/external_reference/';

CREATE TABLE IF NOT EXISTS dbw_pe_platform.bronze.external_payment
USING DELTA LOCATION 'abfss://bronze@pestorage3.dfs.core.windows.net/external_payment/';

SHOW TABLES IN dbw_pe_platform.bronze;

database,tableName,isTemporary
bronze,commitment,false
bronze,external_cash,false
bronze,external_payment,false
bronze,external_position,false
bronze,external_reference,false
bronze,fund,false
bronze,investor,false
bronze,market_price,false
bronze,payment,false
bronze,portfolio_company,false


In [0]:
display(spark.sql("DESCRIBE SCHEMA EXTENDED dbw_pe_platform.silver"))

database_description_item,database_description_value
Catalog Name,dbw_pe_platform
Namespace Name,silver
Comment,"Silver layer - cleaned, deduplicated, DQ-checked, crosswalk-resolved data"
Location,abfss://silver@pestorage3.dfs.core.windows.net/__unitystorage/schemas/7eb9335e-ba2f-4bff-a13e-00a693d792cd
Owner,yash.singh_cs23@gla.ac.in
RootLocation,abfss://silver@pestorage3.dfs.core.windows.net/
Properties,"((unity.catalog.managed.iceberg.defaults.delta.feature.catalogManaged,supported))"
Predictive Optimization,ENABLE (inherited from METASTORE metastore_azure_centralindia)
